# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets, their fields, and columns by `@id`
print("Available record sets (by @id):")
for rs in dataset.record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        print(f"    - {field['@id']}: {field.get('name', '')} (dataType: {field.get('dataType', '')})")
    columns = rs.get('column', [])
    if isinstance(columns, dict):
        columns = [columns]
    if columns:
        print("  Columns:")
        for col in columns:
            print(f"    - {col['@id']}: {col.get('name', '')}")
print()

# Display (as an example) the first record of the first record set
if dataset.record_sets:
    first_recordset_id = dataset.record_sets[0]['@id']
    print(f"First record for record set '{first_recordset_id}':")
    for record in dataset.records(record_set=first_recordset_id):
        print(record)
        break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set, referencing by @id
# Get all record set ids
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# For demonstration, pick the first record set loaded:
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"Columns in record set '{selected_record_set_id}':")
    print(dataframes[selected_record_set_id].columns.tolist())
    print(f"\nSample records from '{selected_record_set_id}':")
    display(dataframes[selected_record_set_id].head())
else:
    print("No record sets with data found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
"""
For demonstration purposes, we pick fields based on common names in proteomics datasets, such as 'Coverage', 'MW', or peptide counts.
You can replace these with exact field @id if you identify them in the schema overview above.
"""

# Choose the loaded record set DataFrame
if dataframes:
    df = dataframes[selected_record_set_id]
    print(f"Columns in use: {df.columns.tolist()}")

    # Try to infer a numeric field (cover common proteomics names)
    candidates = [c for c in df.columns if any(x in c.lower() for x in ["coverage", "mw", "abundance", "peptide", "count"])]
    print(f"Numeric field candidates based on column names: {candidates}")
    if candidates:
        numeric_field = candidates[0]  # Default pick for demonstration
        # Ensure numeric, coerce errors
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    else:
        print("No common numeric field found, using the first column.")
        numeric_field = df.columns[0]

    # Example filtering
    threshold = df[numeric_field].dropna().quantile(0.9)  # e.g., top 10%
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try to choose a reasonable grouping field (protein accession, or sample type, etc.)
    group_candidates = [c for c in df.columns if any(x in c.lower() for x in ["accession", "sample", "class", "type"])]
    if group_candidates:
        group_field = group_candidates[0]
        print(f"Grouping by {group_field}:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        display(grouped_df.head())
    else:
        print("No group field identified.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

if dataframes:
    df = dataframes[selected_record_set_id]
    # Safeguard: if numeric_field is defined
    try:
        plt.figure(figsize=(8,5))
        df[numeric_field].hist(bins=30)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()
        
        # Scatter plot if group_field is found
        if 'group_field' in locals():
            plt.figure(figsize=(10,5))
            grouped_df.plot.bar(x=group_field, y=numeric_field, legend=False, color='skyblue')
            plt.title(f"Mean {numeric_field} by {group_field}")
            plt.ylabel(f"Mean {numeric_field}")
            plt.xticks(rotation=90)
            plt.show()
    except Exception as e:
        print(f"Visualization could not be generated: {e}")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the `Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells` dataset using the `mlcroissant` library, explored its metadata, record sets, and inspected sample data. We performed simple exploratory analysis and visualizations using inferred or detected numeric fields. For deeper insights, refer to the Croissant schema's full field names and adjust the analysis to your research questions.

*You can adapt field selection and filtering based on domain knowledge or specific field `@id`s for robust analysis.*